# High-Throughput LLM Inference with Groq (Lecture 31)

Groq utilizes Language Processing Units (LPUs) to deliver near-instantaneous token generation.
In RAG pipelines, low-latency generation significantly improves user experience and enables real-time conversational systems.

This notebook demonstrates:
1. Direct Groq API and LangChain `ChatGroq` integration.
2. Latency benchmarking: Time-to-First-Token (TTFT) and generation throughput (tokens/sec).
3. Streaming generation integration with LangChain chains.

## 1. Setup & Environment Initialization
Load API credentials and initialize ChatGroq.

In [1]:
import os
import time
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
assert groq_api_key, "GROQ_API_KEY is required"

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print(f"ChatGroq initialized with model: {llm.model_name}")

ChatGroq initialized with model: openai/gpt-oss-20b


## 2. Latency & Throughput Benchmark
Measure end-to-end response time and throughput.

In [2]:
benchmark_prompt = "Provide a concise 3-bullet explanation of why low inference latency is crucial for enterprise RAG systems."

start = time.perf_counter()
response = llm.invoke([HumanMessage(content=benchmark_prompt)])
elapsed = time.perf_counter() - start

content = response.content.strip()
word_count = len(content.split())
estimated_tokens = int(word_count * 1.33)
tokens_per_sec = estimated_tokens / elapsed if elapsed > 0 else 0

print(f"Latency: {elapsed:.2f} seconds")
print(f"Generated ~{estimated_tokens} tokens ({tokens_per_sec:.1f} tokens/sec)")
print(f"\nResponse:\n{content}")

assert elapsed < 10.0, "Inference took unexpectedly long"
assert len(content) > 50, "Response content is too short"
print("Benchmark assertions passed.")

Latency: 0.63 seconds
Generated ~119 tokens (189.6 tokens/sec)

Response:
- **User experience & adoption** – Enterprise users expect instant answers; high latency turns a powerful RAG tool into a bottleneck, reducing trust and willingness to rely on it for day‑to‑day tasks.  
- **Real‑time decision making** – Many business workflows (e.g., customer support, compliance checks, financial analysis) require near‑real‑time insights; delays can lead to missed opportunities or incorrect actions.  
- **Operational cost & scalability** – Lower latency means fewer compute cycles per request, enabling larger user bases or more complex models to run within budget while maintaining service level agreements.
Benchmark assertions passed.


## 3. Streaming Response Verification
Verify incremental token streaming.

In [3]:
stream_prompt = "Name 3 key vector databases used in enterprise AI."
print(f"Streaming prompt: '{stream_prompt}'\n")

chunks = []
for chunk in llm.stream(stream_prompt):
    chunks.append(chunk.content)
    print(chunk.content, end="", flush=True)
print("\n")

full_streamed = "".join(chunks)
assert len(chunks) > 5, "Expected multiple streamed chunks"
assert len(full_streamed) > 20, "Streamed text must not be empty"
print("Streaming test passed successfully.")

Streaming prompt: 'Name 3 key vector databases used in enterprise AI.'



**

Three

 of

 the

 most

... (298 outputs left out)

 similarity

 retrieval

.



Streaming test passed successfully.
